# LangChain Agent Benchmark 02: RAG, Embeddings, Chunking, Vector Stores, and Retrieval

This notebook compares retrieval settings using the text from `docs/paper.pdf`. It uses LangChain's document loader, text splitter, embedding, vector store, and retrieval interfaces directly.
Careful: RAG can only be used on unstructured text, ie. papers, documentation, etc., not tables or databases. RAG adaptations to tabular data exist but are not state of the art RAG. Reliable methods to query structured text follow in notebook 03.


In [ ]:
# Run once per environment. Keep optional provider packages commented until needed.
%pip install -qU langchain langchain-core langchain-community langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters langgraph pandas pydantic pypdf langchain-huggingface sentence-transformers
# Optional local/vector-store packages:
# %pip install -qU faiss-cpu langchain-chroma langchain-qdrant qdrant-client


In [11]:
import os
import time
from pathlib import Path

import pandas as pd

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY before running the examples.")

PDF_PATH = Path("docs/paper.pdf")
bio_docs = PyPDFLoader(str(PDF_PATH)).load()

for doc in bio_docs:
    doc.metadata["source"] = str(PDF_PATH)
    doc.metadata["doc_type"] = "paper_pdf"
    doc.metadata["paper"] = PDF_PATH.name

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 7886b06c-8b74-470f-81a4-f9160a8290a4)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


## 1. Retrieval (RAG)

Retrieval controls whether the answer is generated from model priors alone, retrieved source context, reranked context, or multiple retrieved chunks.
The parameter k controls the top k number of similar chunks retrieved. Less chunks give the mdoel less context so answers are focused and cheaper, more chunks have a better chance of finding the answer but may be dispersive, longer, and more expensive.

In [12]:
question = "What did dexamethasone treatment do to CRISPLD2 expression in airway smooth muscle cells?"
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

response = model.invoke(question)
no_retrieved_answer = response.content

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

retrieved_docs = vector_store.similarity_search(question, k=2)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {question}"
response = model.invoke(prompt)
retrieved_answer = response.content

more_docs = vector_store.similarity_search(question, k=6)
context = "\n\n".join(doc.page_content for doc in more_docs)
prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {question}"
response = model.invoke(prompt)
more_context_answer = response.content

df = pd.DataFrame([
    {"mode": "no_rag", "answer": no_retrieved_answer, "sources": ""},
    {"mode": "rag_k_2", "answer": retrieved_answer, "sources": [doc.metadata for doc in retrieved_docs]},
    {"mode": "rag_k_6", "answer": more_context_answer, "sources": [doc.metadata for doc in more_docs]},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,mode,answer,sources
0,no_rag,"Dexamethasone treatment **increased** CRISPLD2 expression in airway smooth muscle cells. Research has shown that CRISPLD2 is a glucocorticoid-responsive gene, meaning its expression is upregulated by corticosteroids like dexamethasone in these cells.",
1,rag_k_2,I do not know.,"[{'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}]"
2,rag_k_6,"Dexamethasone (DEX) treatment increased CRISPLD2 expression in airway smooth muscle (ASM) cells. This increase was observed at both 4 and 24 hours, and protein levels of CRISPLD2 in ASM cells also increased upon DEX treatment by 1.7-fold at 24 hours. The effect of DEX on CRISPLD2 expression was found to be time and dose dependent.","[{'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 7, 'page_label': '8', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 3, 'page_label': '4', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 6, 'page_label': '7', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 2, 'page_label': '3', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}]"


## 2. Embedding model

Embedding model controls how documents and queries are represented for semantic search, which affects retrieval precision and relevance. The models below differ by size, training objective, language coverage, and domain specialization.


In [17]:
query = "CRISPLD2 glucocorticoid dexamethasone airway smooth muscle"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)

from langchain_huggingface import HuggingFaceEmbeddings

embedding_models = [
    {
        "label": "MiniLM general small",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "why_different": "Small, fast, 384-dim general sentence encoder.",
    },
    # {
    #     "label": "MPNet general larger",
    #     "model_name": "sentence-transformers/all-mpnet-base-v2",
    #     "why_different": "Larger 768-dim general encoder; often stronger but slower.",
    # },
    {
        "label": "Multi-QA retrieval tuned",
        "model_name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "why_different": "Trained on question-answer pairs for semantic search.",
    },
    # {
    #     "label": "Multilingual paraphrase",
    #     "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    #     "why_different": "Multilingual 384-dim model; trades English-only focus for language coverage.",
    # },
    {
        "label": "PubMedBERT biomedical",
        "model_name": "NeuML/pubmedbert-base-embeddings",
        "why_different": "Biomedical literature model tuned on PubMed title-abstract pairs.",
    },
]

rows = []
for spec in embedding_models:
    try:
        start = time.perf_counter()
        embeddings = HuggingFaceEmbeddings(
            model_name=spec["model_name"],
            encode_kwargs={"normalize_embeddings": True},
        )
        dimension = len(embeddings.embed_query("dimension check"))
        vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
        hits = vector_store.similarity_search(query, k=3)
        rows.append({
            "embedding": spec["label"],
            "model_name": spec["model_name"],
            "why_different": spec["why_different"],
            "dimension": dimension,
            "seconds": round(time.perf_counter() - start, 2),
            "top_hits": [doc.page_content[:140] for doc in hits],
            "metadata": [doc.metadata for doc in hits],
        })
    except Exception as exc:
        rows.append({
            "embedding": spec["label"],
            "model_name": spec["model_name"],
            "why_different": spec["why_different"],
            "dimension": None,
            "seconds": None,
            "top_hits": [],
            "metadata": [],
            "error": repr(exc),
        })

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_hits"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)



'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 17f4b972-5b4e-4f0c-818d-b5f58b484d78)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].


,embedding,model_name,why_different,dimension,seconds,top_hits,metadata
0,MiniLM general small,sentence-transformers/all-MiniLM-L6-v2,"Small, fast, 384-dim general sentence encoder.",384,19.460000,"['RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as\na Glucocorticoid Responsive Gene that Modulates\nCytokine Function in Airway Smooth Mu', 'offer a comprehensive view of the effect of a glucocorticoid on the ASM transcriptome and identify CRISPLD2 as an asthma\npharmacogenetics ca', 'Glucocorticoid-induced Changes in Gene Expression of Airway Smooth Muscle\nin Patients with Asthma. Am J Respir Crit Care Med 187: 1076–1084.']","[{'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 11, 'page_label': '12', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}]"
1,Multi-QA retrieval tuned,sentence-transformers/multi-qa-mpnet-base-dot-v1,Trained on question-answer pairs for semantic search.,768,16.180000,"['RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as\na Glucocorticoid Responsive Gene that Modulates\nCytokine Function in Airway Smooth Mu', 'mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway\nsmooth muscle (ASM).', 'doi:10.1371/journal.pone.0099625.g002\nCRISPLD2 Is a Glucocorticoid Responsive Gene in ASM\nPLOS ONE | www.plosone.org 4 June 2014 | Volume 9 ']","[{'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, {'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 3, 'page_label': '4', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}]"
2,PubMedBERT biomedical,NeuML/pubmedbert-base-embeddings,Biomedical literature model tuned on PubMed title-abstract pairs.,768,14.450000,"['mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway\nsmooth muscle (ASM).', 'DEX, and the GSE13168 study found that the differential\nCRISPLD2 expression was strongest when ASM cells were treated\nwith a GC (i.e. flutic', 'Glucocorticoid-induced Changes in Gen

## 3. Chunk size

Chunk size controls the amount of text in each indexed unit, balancing complete context against retrieval specificity.

In notebook 01 section 5, we experimented with context, meaning understanding how much context the LLM could actually see in the prompt to base its answer. That differed from RAG as no documents are searched or retrieved, RAG's role is to automates the context-selection step. 

In [18]:
query = "What RNA-Seq quality control or alignment metrics are reported?"
rows = []

for size in [200, 500, 1000, 2000]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100)
    splits = text_splitter.split_documents(bio_docs)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"chunk_size": size, "n_chunks": len(splits), "top_context": [doc.page_content for doc in hits]})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 4. Chunk overlap

Chunk overlap controls how much text is repeated across adjacent chunks, reducing missing context at boundaries but potentially increasing duplicate retrieval.


In [19]:
query = "What treatment protocol was used for dexamethasone in airway smooth muscle cells?"
rows = []

for overlap in [0, 100, 300]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=overlap)
    splits = text_splitter.split_documents(bio_docs)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=4)
    rows.append({"chunk_overlap": overlap, "n_chunks": len(splits), "top_context": [doc.page_content for doc in hits]})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,chunk_overlap,n_chunks,top_context
0,0,158,"['ASM Cell Culture and GC Treatment for RNA-Seq\nExperiment\nPrimary ASM cells were isolated from four white aborted lung\ntransplant donors with no chronic illness. ASM cell cultivation and\ncharacterization were described previously [52,53]. Passages 4 to 7\nASM cells maintained in Ham’s F12 medium supplemented with\n10% FBS were used in all experiments. For the RNA-Seq and\nqRT-PCR validation experiments, cells from each donor were\ntreated with 1\nmM DEX (Sigma-Aldrich, St. Louis, MO) or', 'smooth muscle (ASM). However, the mechanism by which glucocorticoids suppress inflammation in ASM remains poorly\nunderstood. Using RNA-Seq, a high-throughput sequencing method, we characterized transcriptomic changes in four\nprimary human ASM cell lines that were treated with dexamethasone—a potent synthetic glucocorticoid (1\nmM for\n18 hours). Based on a Benjamini-Hochberg corrected p-value ,0.05, we identified 316 differentially expressed genes,', 'studies have measured the effect of GCs on ASM cells using in vitro\nmodels where human ASM cells were stimulated with dexameth-\nasone or fluticasone [17,18]. Although both were limited by the\ninherent biases of microarrays, these studies identified some genes\ninvolved in the ASM GC response, with one focusing on validating\nthe function of the KLF15 gene in airway hyperresponsiveness [17]\nand the other on the overlap between GC and beta-agonist\nresponse of the ASM [18].', 'greater number of individuals and/or greater sequencing depth\nwill yield additional insight into the ASM transcriptome.\nWhile ASM is a target tissue in the GC treatment of asthma, our\nASM samples were not from asthma patients. Although the study\nby Masuno et al found that there was general concordance\nbetween response to DEX in 16 genes among four control ASM\ncell lines and those of two asthma patients [17], there are likely\nsome differences in the GC response between asthma patients and']"
1,100,179,"['mainstay therapy for asthma because they exert anti-inflammatory effects in multiple lung tissues, including the airway\nsmooth muscle (ASM). However, the mechanism by which glucocorticoids suppress inflammation in ASM remains poorly\nunderstood. Using RNA-Seq, a high-throughput sequencing method, we characterized transcriptomic changes in four\nprimary human ASM cell lines that were treated with dexamethasone—a potent synthetic glucocorticoid (1\nmM for', 'ASM cells were maintained in Ham’s F12 medium supplemented\nwith 24 mM HEPES, 1.7 mM CaCl2, 12 Mm NaOH and 10% FBS.\nFor chemical treatment, cells grown in the above medium were washed\nwith PBS and switched to the serum deprivation medium. A549\nhuman lung epithelial cells were maintained in high-glucose DMEM\nmedium containing 10% FBS. For chemical treatment, A549 cells\nwere washed and switched to DMEM/F12(1:1) medium supplement-\ned with 3% dialyzed FBS. 100 nM DEX (Sigma-Aldrich Corporation,', 'on human experimentation, including obtaining written informed\nconsent for all study participants.\nASM Cell Culture and GC Treatment for RNA-Seq\nExperiment\nPrimary ASM cells were isolated from four white aborted lung\ntransplant donors with no chronic illness. ASM cell cultivation and\ncharacterization were described previously [52,53]. Passages 4 to 7\nASM cells maintained in Ham’s F12 medium supplemented with\n10% FBS were used in all experiments. For the RNA-Seq and', 'between response to DEX in 16 genes among four control ASM\ncell lines and those of two asthma patients [17], there are likely\nsome differences in the GC response between asthma patients and\nindividuals without asthma. Further studies that include ASM\nfrom asthma patients may help clarify such differences. Finally, it\nis known that the response to GCs changes in time. For example,\nthe Masuno et al study compared the ASM GC response at both 4']"
2,300,341,"['Experiment\nPrimary ASM cells were isolated from four white aborted lung\ntranspla

## 5. Vector store

Vector store controls where embeddings are stored and queried, affecting speed, usability, persistence, and scalability.


In [ ]:
query = "CRISPLD2 dexamethasone cytokine IL6 IL8"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
rows = []

start = time.perf_counter()
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
hits = vector_store.similarity_search(query, k=3)
rows.append({"store": "InMemoryVectorStore", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})

try:
    from langchain_chroma import Chroma

    start = time.perf_counter()
    vector_store = Chroma.from_documents(documents=splits, embedding=embeddings, collection_name="bio_benchmark")
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "Chroma", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})
except Exception as exc:
    print("Chroma skipped:", exc)

try:
    from langchain_community.vectorstores import FAISS

    start = time.perf_counter()
    vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "FAISS", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})
except Exception as exc:
    print("FAISS skipped:", exc)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_metadata"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


Following section: 
Strategies for selecting which documents (or chunks) to return from a vector database after embedding a query. Choose the based on whether you want to optimize only for relevance or also for diversity

## 6. Similarity search

Similarity search retrieves chunks closest to the query embedding, usually favoring relevance over diversity.


In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

vector_store.similarity_search("CRISPLD2 IL1 beta IL6 IL8 knockdown", k=3)


[Document(id='2649a248-cb06-4957-8eb3-873fca935cef', metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows); modified using iText 5.0.3 (c) 1T3XT BVBA', 'creator': '3B2 Total Publishing System 7.51n/W', 'creationdate': '2014-06-03T10:42:37+08:00', 'title': 'pone.0099625 1..13', 'moddate': '2014-06-04T01:26:28-07:00', 'source': 'docs/paper.pdf', 'total_pages': 13, 'page': 4, 'page_label': '5', 'doc_type': 'paper_pdf', 'paper': 'paper.pdf'}, page_content='the proinflammatory cytokine IL1 b (5 ng/mL for 24 h) increased\nCRISPLD2 mRNA by 10.4-fold and protein levels by 1.9-fold\n[Figure 3C and 3D], suggesting that CRISPLD2 is not only GC-\ninducible but also immuno-responsive. We next performed\nknockdown experiments to assess whether CRISPLD2 modulates\nIL1b-induced cytokine responses using a single ASM cell line.\nBecause IL1 b acts as an important mediator of inflammatory\nresponses by activating other cytokines, we investigated the role of'),
 Document(id='180bac4a-ce74-4efe-89e2-991

## 7. MMR retrieval

MMR (Maximal Marginal Relevance) returns both relevant and diverse chunks


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

vector_store.max_marginal_relevance_search("RNA-Seq transcriptome profiling dexamethasone glucocorticoid response", k=3)


## 8. Metadata filtering

Metadata filtering restricts retrieval to documents with selected labels, such as source file or PDF page ranges.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

first_page_hits = [
    doc for doc in vector_store.similarity_search("abstract CRISPLD2 dexamethasone", k=8)
    if doc.metadata.get("page") == 0
]
early_results_hits = [
    doc for doc in vector_store.similarity_search("RNA-Seq results differentially expressed genes", k=8)
    if 1 <= doc.metadata.get("page", -1) <= 4
]
paper_pdf_hits = [
    doc for doc in vector_store.similarity_search("glucocorticoid airway smooth muscle", k=8)
    if doc.metadata.get("doc_type") == "paper_pdf"
]

df = pd.DataFrame([
    {"filter": "first PDF page", "hits": [doc.metadata for doc in first_page_hits]},
    {"filter": "early results pages", "hits": [doc.metadata for doc in early_results_hits]},
    {"filter": "paper PDF source", "hits": [doc.metadata for doc in paper_pdf_hits]},
])
df.style.set_properties(
    subset=["hits"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 9. Top-k retrieval

Top-k retrieval controls how many chunks are passed to the generator, affecting evidence coverage, distraction, token cost, and answer length.


In [ ]:
query = "What were the main RNA-Seq findings about CRISPLD2 and cytokine regulation?"
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
rows = []

for k in [2, 5, 10]:
    retrieved_docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {query}"
    response = model.invoke(prompt)
    rows.append({"k": k, "n_sources": len(retrieved_docs), "sources": [doc.metadata for doc in retrieved_docs], "answer": response.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)
